In [3]:
from pathlib import Path
import sys
import cv2
import numpy as np

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "utils" / "analyze_water_cooling.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Не найден корень проекта с каталогом src")
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.analyze_water_cooling import load_video_memmap

import json

In [4]:
metadata_dir = PROJECT_ROOT / "data/honeycomb/metadata/videos"
videos_dir = PROJECT_ROOT / "data/raw"
output_dir = PROJECT_ROOT / "data/processed/cells"
output_dir.mkdir(parents=True, exist_ok=True)

# load_video_memmap не загружает весь 3.5-гигабайтный MAT-файл в RAM.
DEFAULT_FPS = 10.0

for metadata_path in sorted(metadata_dir.glob("*.json")):
    video_name = metadata_path.stem
    video_path = videos_dir / f"{video_name}.mat"

    with metadata_path.open(encoding="utf-8") as metadata_file:
        metadata = json.load(metadata_file)

    video, fps = load_video_memmap(video_path, "data")
    height, width, frame_count = video.shape
    output_fps = fps if fps > 0 else DEFAULT_FPS

    for cell in metadata.get("cells", []):
        class_name = str(cell["class_name"])
        x1, y1, x2, y2 = (int(value) for value in cell["bbox_xyxy"])
        x1, x2 = sorted((max(0, x1), min(width - 1, x2)))
        y1, y2 = sorted((max(0, y1), min(height - 1, y2)))
        crop_width, crop_height = x2 - x1 + 1, y2 - y1 + 1
        if crop_width <= 0 or crop_height <= 0:
            raise ValueError(f"Некорректный bbox для {video_name}/{class_name}")

        output_path = output_dir / f"{video_name}_{class_name}.mp4"
        writer = cv2.VideoWriter(
            str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), output_fps,
            (crop_width, crop_height), isColor=False,
        )
        if not writer.isOpened():
            raise RuntimeError(f"Не удалось открыть VideoWriter: {output_path}")

        try:
            for frame_index in range(frame_count):
                crop = np.asarray(video[y1:y2 + 1, x1:x2 + 1, frame_index])
                crop = np.nan_to_num(crop, nan=0.0, posinf=0.0, neginf=0.0)
                low, high = np.percentile(crop, (1, 99))
                if high <= low:
                    frame_u8 = np.zeros(crop.shape, dtype=np.uint8)
                else:
                    frame_u8 = np.clip((crop - low) * 255.0 / (high - low), 0, 255).astype(np.uint8)
                writer.write(frame_u8)
        finally:
            writer.release()

        print(f"Готово: {output_path} ({frame_count} кадров)")


Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water80.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water100.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water120.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water20.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water40.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water1_water60.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water2_water60.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water2_water40.mp4 (3000 кадров)
Готово: /home/votter/projects/honeycomb-water-detection/data/processed/cells/water2_water20.mp4 (3000 кадров)
Готово: 

In [ ]:
scene_path = Path("data/raw/water1.mp4")
